# Lab 5 — FP16 vs INT4-AWQ: quality, latency, and what it costs

**The claim you should be able to make when you finish:** *"I can tell you what
a 4-bit quant buys and what it costs on your traffic, with a measurement rather
than a vibe — and I know which measurement people skip."*

The skipped measurement is distributional drift. Task accuracy on 60 prompts is
noisy and easy to pass; the sensitive question is *how far did the next-token
distribution move*. A quant that keeps top-1 agreement with the fp16 reference
above ~0.95 is very unlikely to change behaviour on traffic you have not tested.
One that drops to 0.8 will surprise you in production even if your benchmark
score held.

### What runs where

| variant | T4 | L4 / A100 |
|---|---|---|
| FP16 baseline | yes | yes |
| INT4 AWQ | yes | yes |
| FP8 (weights or KV) | **no** — needs Ada sm_89+ | yes |

Run the first two on the free tier. The FP8 row fills itself in when you rerun
on an L4; the notebook detects the card and skips what it cannot do.

In [ ]:
# Cell 1 — idempotent bootstrap.
REPO   = "https://github.com/lsgrep/serv.git"
BRANCH = "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("matplotlib", "pandas", "httpx")
pip("vllm")

import servlab
env = servlab.notebook_setup()

## 1. Predict the savings before measuring the damage

Quantisation moves two things at once, and conflating them is the most common
confusion here:

* **Weight quantisation** shrinks the weights. That is memory *and* bandwidth —
  a decode step reads the weights every step, so 4-bit weights make decode
  roughly proportionally faster at small batch.
* **KV quantisation** shrinks the cache. That is concurrency: FP8 KV doubles the
  sequences you can hold at a given context length.

They are independent settings. An INT4-AWQ model with an fp16 KV cache is the
common default, and it improves throughput without improving concurrency much.

In [ ]:
from servlab import napkin as nk

SPEC = nk.MODELS["qwen2.5-7b"]
CARD = "T4" if "T4" in env.gpu_name else ("L4" if "L4" in env.gpu_name else "A100-40GB")

print(f"{'variant':<26}{'weights':>12}{'seqs @2k':>11}{'decode b=8':>13}")
rows = [
    ("fp16 weights, fp16 KV", "fp16", "fp16"),
    ("int4 AWQ,     fp16 KV", "int4", "fp16"),
    ("int4 AWQ,     fp8 KV",  "int4", "fp8"),
]
for label, w, kv in rows:
    wb = nk.awq_weight_bytes(SPEC) if w == "int4" else nk.weight_bytes(SPEC, w)
    seqs = nk.max_concurrent_sequences(CARD, SPEC, 2048, weight_dtype=w, kv_dtype=kv)
    tps = nk.decode_tokens_per_s(CARD, SPEC, batch=8, ctx_len=2048,
                                 weight_dtype=w, kv_dtype=kv)
    print(f"{label:<26}{nk.human_bytes(wb):>12}{max(seqs,0):>11,.0f}{tps:>12,.0f}/s")

print(f"\nA 7B model in fp16 does not fit on a 16 GB card at all. In AWQ it does,")
print(f"with room for KV. That is the decision quantisation is usually making —")
print(f"not 'a bit faster', but 'runs at all on the hardware you have'.")

## 2. Serve the FP16 baseline and measure it

Same model family, two checkpoints. Use a pre-quantised AWQ checkpoint from the
Hub rather than quantising yourself — AWQ calibration takes longer than this
whole lab, and the packaged checkpoints are what you would deploy anyway.

Everything below runs one variant at a time: start server, eval, bench, stop,
free VRAM. On a 16 GB card there is no room for two.

In [ ]:
from servlab.serve import VLLMServer
from servlab.loadgen import run_load
from servlab.stats import summarize
from servlab.evalkit import arithmetic_cases, format_cases, run_eval, openai_generator
import gc, torch, time

FP16_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"          # fits fp16 on a T4 with KV room
AWQ_MODEL  = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"      # same family, 4-bit
# On an L4/A100, step up to the 7B pair — the effect is larger and more honest:
# FP16_MODEL = "Qwen/Qwen2.5-7B-Instruct"
# AWQ_MODEL  = "Qwen/Qwen2.5-7B-Instruct-AWQ"

CASES = arithmetic_cases(n=40, seed=7) + format_cases(n=20, seed=7)
BASE  = "http://localhost:8000"
results, latencies, sizes = [], {}, {}
print(f"{len(CASES)} eval cases")

In [ ]:
def measure(name, model_id, quantization=None, kv_dtype="auto", max_model_len=2048):
    server = VLLMServer(model_id, port=8000, max_model_len=max_model_len,
                        gpu_memory_utilization=0.85, enforce_eager=True,
                        quantization=quantization,
                        extra=(["--kv-cache-dtype", kv_dtype] if kv_dtype != "auto" else []),
                        log_path=f"runs/lab5-{name}.log").start()
    try:
        # Quality: greedy, so any difference is the quantisation, not sampling.
        res = run_eval(CASES, openai_generator(BASE, model_id), name=name)
        print(res)

        # Speed: one closed-loop level, same workload for every variant.
        bench = summarize(run_load(BASE, model_id, concurrency=8, duration=30,
                                   prompt_tokens=256, max_tokens=128),
                          slo_ttft=1.0, slo_tpot=0.05)
        print(bench)

        # Capacity: what the engine says it got, not what we hoped for.
        kv_line = next((l for l in server.tail(200).splitlines() if "KV cache" in l), "")
        print(kv_line.strip() or "(KV cache line not found in log)")

        results.append(res)
        latencies[name] = {"output_throughput": bench.output_throughput,
                           "ttft_p50": bench.ttft.get("p50"),
                           "tpot_p50": bench.tpot.get("p50")}
        return res, bench
    finally:
        server.stop()
        gc.collect(); torch.cuda.empty_cache(); time.sleep(5)

In [ ]:
fp16_res, fp16_bench = measure("fp16", FP16_MODEL)

In [ ]:
awq_res, awq_bench = measure("int4-awq", AWQ_MODEL, quantization="awq")

In [ ]:
# FP8 needs Ada (sm_89) or newer. On a T4 this cell prints why it is skipped.
if env.supports_fp8:
    fp8_res, fp8_bench = measure("int4-awq + fp8 KV", AWQ_MODEL,
                                 quantization="awq", kv_dtype="fp8")
else:
    print(f"skipping FP8: {env.gpu_name} is sm_{env.capability[0]}{env.capability[1]}, "
          "FP8 needs sm_89+ (L4, Ada, Hopper).")
    print("Switch the Colab runtime to an L4 and rerun this notebook to fill the row.")

## 3. The table that decides

Quality, speed, memory and cost together. Any one of them alone will talk you
into the wrong answer:

* accuracy alone → "INT4 is free" (it is not, and 60 prompts cannot tell you)
* throughput alone → "always quantise" (ignores the tail-quality risk)
* cost alone → "buy the biggest card" (ignores whether the model even fits)

In [ ]:
from servlab.evalkit import compare_table
from servlab.plots import bar_compare

usd = nk.GPUS[CARD].usd_per_hour
print(compare_table(results, latencies=latencies, usd_per_hour=usd))
print(f"\n(cost assumes ${usd:.2f}/h for {CARD} — edit servlab.napkin.GPUS to your real price)")

bar_compare([r.name for r in results], [r.accuracy * 100 for r in results],
            title="task accuracy", ylabel="%", fmt="{:.0f}%")
bar_compare([r.name for r in results],
            [latencies[r.name]["output_throughput"] for r in results],
            title="output tokens/s at concurrency 8", ylabel="tok/s")

In [ ]:
# Where did the quantised model actually differ? Read the disagreements, do not
# just look at the delta — a 2-point accuracy drop that is all in one tag is a
# different finding from one spread evenly.
def disagreements(a, b, limit=8):
    out = []
    for x, y in zip(a.outputs, b.outputs):
        if x["ok"] != y["ok"]:
            out.append((x["prompt"][:70], x["expected"], x["got"][:40].strip(),
                        y["got"][:40].strip(), "fp16" if x["ok"] else "awq"))
    print(f"{len(out)} of {len(a.outputs)} cases disagree\n")
    for p, exp, g1, g2, winner in out[:limit]:
        print(f"  {p}\n    expected {exp!r}   fp16 {g1!r}   awq {g2!r}   -> {winner} correct\n")

disagreements(fp16_res, awq_res)

print("per-tag breakdown:")
for name, res in (("fp16", fp16_res), ("awq", awq_res)):
    print(f"  {name:<6}", {k: f"{v['correct']}/{v['n']}" for k, v in sorted(res.by_tag.items())})

## 4. The measurement people skip: distributional drift

Accuracy over 60 prompts has a standard error of roughly 6 points. A 3-point
"drop" is noise. To detect real damage you need something with far more signal
per prompt: compare the *distributions* the two models produce over the same
prefixes.

**Top-1 agreement** — the fraction of positions where both models would pick the
same next token — is the single most legible number, and one prompt gives you
hundreds of samples of it. Rule of thumb for these labs:

| top-1 agreement | reading |
|---|---|
| > 0.95 | behaviourally interchangeable on this kind of text |
| 0.85 - 0.95 | drift you should measure on your own traffic before shipping |
| < 0.85 | expect visible behaviour change |

Collect greedy continuations from both variants over the same prompts, then
compare token by token. Greedy decoding makes this deterministic, so any
divergence is the quantisation.

In [ ]:
# Re-serve each variant briefly to collect greedy continuations on shared prompts.
from servlab.evalkit import top1_agreement
import json, urllib.request

PROBE_PROMPTS = [
    "Explain what a KV cache is and why serving systems need one.",
    "Write a Python function that reverses a linked list.",
    "Summarise the tradeoff between throughput and latency in one paragraph.",
    "List three reasons a GPU server might have high tail latency.",
    "What is grouped-query attention?",
]

def greedy_tokens(model_id, quantization=None, max_tokens=96):
    server = VLLMServer(model_id, port=8000, max_model_len=2048,
                        gpu_memory_utilization=0.85, enforce_eager=True,
                        quantization=quantization,
                        log_path=f"runs/lab5-probe.log").start()
    try:
        gen = openai_generator(BASE, model_id, max_tokens=max_tokens, temperature=0.0)
        return [gen(p) for p in PROBE_PROMPTS]
    finally:
        server.stop()
        gc.collect(); torch.cuda.empty_cache(); time.sleep(5)

fp16_texts = greedy_tokens(FP16_MODEL)
awq_texts  = greedy_tokens(AWQ_MODEL, quantization="awq")

In [ ]:
# Token-level agreement, using the shared tokenizer so positions line up.
from transformers import AutoTokenizer

tk = AutoTokenizer.from_pretrained(FP16_MODEL)
scores = []
for a, b in zip(fp16_texts, awq_texts):
    ta, tb = tk.encode(a), tk.encode(b)
    n = min(len(ta), len(tb))
    scores.append(top1_agreement(ta[:n], tb[:n]))

for p, s, a, b in zip(PROBE_PROMPTS, scores, fp16_texts, awq_texts):
    print(f"{s:6.1%}  {p[:56]}")
    if s < 0.95:
        # Where they first diverge is usually more informative than the score.
        ta, tb = tk.encode(a), tk.encode(b)
        i = next((j for j in range(min(len(ta), len(tb))) if ta[j] != tb[j]), None)
        if i is not None:
            print(f"         diverges at token {i}: "
                  f"{tk.decode(ta[max(0,i-6):i+1])!r} -> fp16 {tk.decode([ta[i]])!r} "
                  f"vs awq {tk.decode([tb[i]])!r}")

print(f"\nmean top-1 agreement: {sum(scores)/len(scores):.1%}")
print("\nGreedy decoding is deterministic, so every divergence here is the quant.")
print("Note that agreement falls naturally after the first divergence — the two")
print("models are now continuing different texts. The position of the first")
print("divergence carries more signal than the aggregate.")

## 5. What to be able to say afterwards

1. **Weight quantisation and KV quantisation are different levers** — one buys
   bandwidth and memory, the other buys concurrency.
2. **AWQ is not "4 bits per parameter."** Group scales and zero-points put it
   nearer 4.5, which is why the checkpoint is bigger than params/2 and why the
   napkin math should use `awq_weight_bytes`.
3. **A 60-prompt eval cannot detect a 3-point difference.** Say the standard
   error out loud; it is the fastest way to show you understand the measurement.
4. **Top-1 agreement is the cheap sensitive test**, and the first divergence
   point is more informative than the average.
5. **The decision is usually "fits or does not fit."** Quantisation's headline
   benefit is running a model class you otherwise could not host, and the
   quality question is whether that trade is acceptable — not whether it is free.

**Next:** lab 6 leaves CUDA entirely and looks at what serving is like on a TPU.